In [3]:
import tensorflow as tf
import numpy as np
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"Tf version: {tf.__version__}\n")

Tf version: 2.21.0



In [4]:
t = tf.constant([[1., 2., 3.],[4., 5., 6.]])
print(t)
print(t.shape)
print(t.dtype)
print()
print(t+10)
print(tf.square(t))
print(t @ tf.transpose(t))
print()
a = np.array([1,2,3])
t_from_np = tf.constant(a)
print(t_from_np)
print(t_from_np.numpy())
print()
t_float = tf.constant(2.0, dtype=tf.float32)
t_int = tf.constant(3, dtype=tf.int32)

# print('Попробую сломать типизацию:')
# try: 
#     res = t_float + t_int
# except TypeError as e:
#     print('Ошибка типов!')
#     print(e)

res_fix = t_float + tf.cast(t_int, tf.float32)
print(res_fix.numpy())

v = tf.Variable([[1., 2.], [3., 4.]])
print("\nИзначальная переменная v:\n", v.numpy())
v.assign(v * 2)
print("Переменная v после v.assign(v * 2):\n", v.numpy())
v[0, 1].assign(42.0)
print("Переменная v после точечного изменения:\n", v.numpy())

tf.Tensor(
[[1. 2. 3.]
 [4. 5. 6.]], shape=(2, 3), dtype=float32)
(2, 3)
<dtype: 'float32'>

tf.Tensor(
[[11. 12. 13.]
 [14. 15. 16.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[ 1.  4.  9.]
 [16. 25. 36.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[14. 32.]
 [32. 77.]], shape=(2, 2), dtype=float32)

tf.Tensor([1 2 3], shape=(3,), dtype=int64)
[1 2 3]

5.0

Изначальная переменная v:
 [[1. 2.]
 [3. 4.]]
Переменная v после v.assign(v * 2):
 [[2. 4.]
 [6. 8.]]
Переменная v после точечного изменения:
 [[ 2. 42.]
 [ 6.  8.]]


In [5]:
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

def myself_huber_loss(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1.0
    squared_loss = tf.square(error) / 2
    linear_loss = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

model = keras.models.Sequential([
    keras.layers.Input(shape=X_train.shape[1:]),
    keras.layers.Dense(30, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dense(1)
])

model.compile(loss=myself_huber_loss, optimizer='adam', metrics=['mae'])
print('Обучение...')
model.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))

Обучение...
Epoch 1/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.5211 - mae: 0.8774 - val_loss: 0.3442 - val_mae: 0.6565
Epoch 2/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2548 - mae: 0.5612 - val_loss: 0.2579 - val_mae: 0.5516
Epoch 3/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2146 - mae: 0.5050 - val_loss: 0.2034 - val_mae: 0.4820
Epoch 4/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1948 - mae: 0.4760 - val_loss: 0.1764 - val_mae: 0.4489
Epoch 5/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1832 - mae: 0.4584 - val_loss: 0.1679 - val_mae: 0.4355


In [6]:
model.save('model_with_custom_loss.keras')
print('Модель сохранена на диск!')

loaded_model = keras.models.load_model(
    'model_with_custom_loss.keras',
    custom_objects = {'myself_huber_loss': myself_huber_loss}
)

print('Модель загружена!')
predictions = loaded_model.predict(X_test[:3])
print(f"Предсказание: {predictions.ravel()}")

Модель сохранена на диск!
Модель загружена!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
Предсказание: [0.5171778 1.7517958 3.0721972]


In [7]:
# КАСТОМНАЯ АКТИВАЦИЯ
def my_softplus(z):
    return tf.math.log(1.0 + tf.math.exp(z))

# КАСТОМНЫЙ ИНИЦИАЛИЗАТОР
def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2.0 / (shape[0] + shape[1]))
    return tf.random.normal(shape, mean=1.0, stddev=stddev, dtype=dtype)

# КАСТОМНЫЙ РЕГУЛЯРИЗАТОР
def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(weights)) * 0.01

# КАСТОМНОЕ ОГРАНИЧЕНИЕ
def my_positive_constraint(weights):
    return tf.where(weights < 0., 0., weights)

# СОБИРАЕМ ВСЁ В ОДНУ МОДЕЛЬ
model_custom_everything = keras.models.Sequential([
    keras.layers.Input(shape=X_train.shape[1:]),
    keras.layers.Dense(30, 
                       activation=my_softplus,
                       kernel_initializer=my_glorot_initializer,
                       kernel_regularizer=my_l1_regularizer,
                       kernel_constraint=my_positive_constraint),
    keras.layers.Dense(1)
])

print("Модель, где кастомное ВООБЩЕ ВСЁ, успешно собрана!")

model_custom_everything.compile(loss=myself_huber_loss, optimizer="adam")
model_custom_everything.fit(X_train, y_train, epochs=2, verbose=1)


Модель, где кастомное ВООБЩЕ ВСЁ, успешно собрана!
Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: nan
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: nan


In [8]:
@keras.utils.register_keras_serializable()
class myselfHyberMetric(keras.metrics.Metric):
    def __init__(self, threshold=1.0, name="myself_hyber_metric", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.total = self.add_weight(shape=(), initializer='zeros', name='total')
        self.count = self.add_weight(shape=(), initializer='zeros', name='count')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, dtype=tf.float32)
        y_pred = tf.cast(y_pred, dtype=tf.float32)
        
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2
        linear_loss = self.threshold * (tf.abs(error) - self.threshold / 2)
        base_metric = tf.where(is_small_error, squared_loss, linear_loss)
        
        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, dtype=tf.float32)
            base_metric = tf.multiply(base_metric, sample_weight)
            num_elements = tf.reduce_sum(sample_weight)
        else:
            num_elements = tf.cast(tf.size(y_true), tf.float32)

        self.total.assign_add(tf.reduce_sum(base_metric))
        self.count.assign_add(num_elements)

    def result(self):
        return self.total / self.count

    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)


In [9]:
model_custom_everything.compile(loss=myself_huber_loss, optimizer="adam", metrics=[myselfHyberMetric(threshold=1.0)])
model_custom_everything.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: nan - myself_hyber_metric: nan - val_loss: nan - val_myself_hyber_metric: nan
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: nan - myself_hyber_metric: nan - val_loss: nan - val_myself_hyber_metric: nan


In [10]:
class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, batch_input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=[batch_input_shape[-1], self.units],
            initializer="glorot_normal"
        )
        self.bias = self.add_weight(
            name="bias",
            shape=[self.units],
            initializer="zeros" 
        )
        
        super().build(batch_input_shape)

    def call(self, X):
        result = X @ self.kernel + self.bias
        if self.activation is not None:
            return self.activation(result)
        return result

print("Класс кастомного слоя MyDense успешно создан!")


model_custom_layer = keras.models.Sequential([
    keras.layers.Input(shape=X_train.shape[1:]),
    MyDense(30, activation="relu"),
    MyDense(1)                    
])

Класс кастомного слоя MyDense успешно создан!


In [11]:
model_custom_layer.compile(loss="mse", optimizer="adam")
model_custom_layer.fit(X_train, y_train, epochs=2, verbose=1)

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.6712
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6140


In [12]:
class MyCustomModel(keras.Model):
    def __init__(self, output_units, **kwargs):
        super().__init__(**kwargs)
        self.hidden1 = MyDense(30, activation="relu")
        self.hidden2 = MyDense(30, activation="relu")
        self.out_layer = MyDense(output_units)

    def call(self, inputs):
        hidden1 = self.hidden1(inputs)
        hidden2 = self.hidden2(hidden1)
        output = self.out_layer(hidden2)
        return output

print("Класс кастомной архитектуры MyCustomModel успешно создан!")

model_custom_architecture = MyCustomModel(output_units=1)

model_custom_architecture.compile(loss="mse", optimizer="adam")
model_custom_architecture.fit(X_train, y_train, epochs=2, verbose=1)

Класс кастомной архитектуры MyCustomModel успешно создан!
Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1.3333
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4272


In [13]:
class ModelWithInternalLoss(keras.Model):
    def __init__(self, output_units, **kwargs):
        super().__init__(**kwargs)
        self.hidden = keras.layers.Dense(30, activation="relu", kernel_initializer="he_normal")
        self.out_layer = keras.layers.Dense(output_units)

    def call(self, inputs):
        hidden_output = self.hidden(inputs)
        internal_loss = tf.reduce_mean(tf.square(hidden_output)) * 0.05
        self.add_loss(internal_loss)
        return self.out_layer(hidden_output)

print("Модель со скрытыми внутренними потерями создана!")

model_internal = ModelWithInternalLoss(output_units=1)
model_internal.compile(loss="mse", optimizer="adam")
model_internal.fit(X_train, y_train, epochs=2, verbose=1)


Модель со скрытыми внутренними потерями создана!
Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.9366
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6875
